# 08 · Validación de la inferencia de punta a punta — Grupo 17 · cierre de la fase de modelado
**Entorno CPU** (así se medirá en producción). Requiere en Drive: `src/inference.py`, `contracts/` (esquema + golden),
`tests/test_inference.py` y los artefactos de `models/`.

| Prueba | Criterio de aceptación |
|---|---|
| P1 · Suite completa (pytest) | 366 pruebas en verde en este entorno |
| P2 · Fidelidad: `inference.py` desde el correo vs. probabilidades del 07 | `texto_modelo` idéntico ≥ 99 %; misma clase ≥ 98 %; \|ΔP(phishing)\| mediana < 0,01 |
| P3 · Contrato: 300 salidas reales validadas contra el JSON Schema | 100 % válidas; 0 IOCs vivos |
| P4 · Latencia en CPU de punta a punta | media y p95 < 1.000 ms |
| P5 · Revisión manual de phishing_pot (si ya existe el CSV revisado) | proporción por veredicto con IC 95 % |

In [2]:
# Celda 1: Entorno
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np, pandas as pd
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
sys.path.insert(0, str(BASE / "src"))
for f in ("src/inference.py", "contracts/contrato_v1.schema.json", "models/fusion_v1.joblib", "models/distilbert_v1"):
    assert (BASE / f).exists(), f"Falta {f}"
PROC, RES = BASE / "data/processed", BASE / "results"
print(ENV, BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
colab /content/drive/MyDrive/PI_G17


In [3]:
# Celda 2: P1 suite completa
%pip install -q jsonschema pytest
carpeta = next(d for d in (BASE / "tests", BASE / "test") if d.exists())
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(carpeta)], capture_output=True, text=True, cwd=BASE)
P1 = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-400:]
print(P1)

363 passed, 1 skipped in 9.06s


In [4]:
# Celda 3: Carga del detector real
import importlib, inference
importlib.reload(inference)
t0 = time.time(); DET = inference.Detector.desde_directorio(BASE); CARGA_S = round(time.time() - t0, 1)
print("variante", DET.variante, "| umbral", round(DET.umbral, 4), "| carga", CARGA_S, "s")

Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

variante V3 | umbral 0.039 | carga 31.3 s


In [5]:
# Celda 4: P2 fidelidad y P3 contrato sobre 300 correos de test
import jsonschema, joblib
ESQ = jsonschema.Draft202012Validator(json.loads((BASE / "contracts/contrato_v1.schema.json").read_text()),
                                      format_checker=jsonschema.FormatChecker())
V1 = pd.read_parquet(PROC / "emails_v1.parquet", columns=["id", "hash", "grupo", "y", "split", "texto_modelo"])
V0 = pd.read_parquet(BASE / "data/interim/emails_v0.parquet", columns=["hash", "texto_defang"])
T = pd.read_parquet(PROC / "text_scores_distilbert_v1.parquet"); U = pd.read_parquet(PROC / "url_scores_v1.parquet")
M = (V1[V1["split"] == "test"].sample(frac=1, random_state=17).groupby("grupo").head(60)
       .merge(V0, on="hash").merge(T, on="id").merge(U, on="id", how="left"))
fus = joblib.load(BASE / "models/fusion_v1.joblib"); base = joblib.load(BASE / "models/baseline_tfidf_lr_v1.joblib")
DOM = {"nazario_anual": ("monkey.org", "monkey")}                  # = CONFIG de 03
filas, invalidas, vivas = [], 0, 0
for _, r in M.iterrows():
    DET.dominios_propios = DOM.get(r["grupo"], ())
    s = DET.analizar("", r["texto_defang"])
    try: ESQ.validate(s)
    except jsonschema.ValidationError: invalidas += 1
    vivas += bool(inference.ioc.iocs_vivos(json.dumps(s)))
    Pd = np.array([[r["p_legitimo"], r["p_spam"], r["p_phishing"]]]); Pb = base.predict_proba([r["texto_modelo"][:5000]])
    n = 0 if pd.isna(r["n_urls_eval"]) else int(r["n_urls_eval"])
    Pref = fus["modelo"].predict_proba(inference.features_meta(fus["variante"], Pd, Pb,
                                       None if pd.isna(r["url_b_max"]) else r["url_b_max"],
                                       None if pd.isna(r["url_b_media"]) else r["url_b_media"], n))[0]
    clase_ref = 2 if Pref[2] >= fus["umbral_fpr1"] else int(np.argmax(Pref[:2]))
    tm = inference.dp.texto_para_modelo(inference.ioc.defang_texto("\n" + r["texto_defang"]), DOM.get(r["grupo"], ()))
    filas.append({"grupo": r["grupo"], "texto_igual": tm == r["texto_modelo"], "n_urls_igual": s["ramas"]["url"]["n_urls"] == n,
                  "dP": abs(s["clasificacion"]["probabilidades"]["phishing"] - Pref[2]),
                  "clase_igual": s["clasificacion"]["clase"] == inference.CLASES[clase_ref],
                  "clase_ok": s["clasificacion"]["clase"] == inference.CLASES[r["y"]], "lat": s["latencia_ms"]})
F = pd.DataFrame(filas)
P2 = {"n": len(F), "pct_texto_modelo_identico": round(100 * F["texto_igual"].mean(), 1), "pct_n_urls_identico": round(100 * F["n_urls_igual"].mean(), 1),
      "pct_misma_clase_que_07": round(100 * F["clase_igual"].mean(), 1), "dP_mediana": round(float(F["dP"].median()), 4),
      "dP_p95": round(float(F["dP"].quantile(.95)), 4), "acierto_vs_etiqueta_pct": round(100 * F["clase_ok"].mean(), 1)}
P3 = {"salidas": len(F), "invalidas_contra_esquema": invalidas, "con_ioc_vivo": vivas}
P2, P3

({'n': 322,
  'pct_texto_modelo_identico': np.float64(100.0),
  'pct_n_urls_identico': np.float64(97.2),
  'pct_misma_clase_que_07': np.float64(100.0),
  'dP_mediana': 0.0,
  'dP_p95': 0.0,
  'acierto_vs_etiqueta_pct': np.float64(99.7)},
 {'salidas': 322, 'invalidas_contra_esquema': 0, 'con_ioc_vivo': 0})

In [6]:
# Celda 5: P4 latencia de punta a punta en CPU
DET.dominios_propios = ()
lat = []
for t in M["texto_defang"].head(100):
    t0 = time.perf_counter(); DET.analizar("", t); lat.append(1000 * (time.perf_counter() - t0))
P4 = {"n": len(lat), "media_ms": round(float(np.mean(lat)), 1), "p95_ms": round(float(np.percentile(lat, 95)), 1),
      "max_ms": round(float(np.max(lat)), 1), "cumple_<1000ms": bool(np.percentile(lat, 95) < 1000)}
P4

{'n': 100,
 'media_ms': 200.3,
 'p95_ms': 348.2,
 'max_ms': 387.0,
 'cumple_<1000ms': True}

In [7]:
# Celda 6: P5 revisión manual de phishing_pot (opcional)
def wilson(k, n, z=1.96):
    if n == 0: return (None, None)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(100 * (c - h), 1), round(100 * (c + h), 1))
rev_path = next((p for p in (BASE / "data/eda/revision_manual_pot_revisado.csv", BASE / "data/eda/revision_manual_pot.csv") if p.exists()), None)
P5 = "pendiente: no hay veredictos"
if rev_path is not None:
    R = pd.read_csv(rev_path); v = R["veredicto_revisor"].fillna("").str.strip().str.lower()
    v = v[v != ""]
    if len(v):
        P5 = {"archivo": rev_path.name, "revisados": int(len(v)),
              "por_veredicto": {k: {"n": int(c), "pct": round(100 * c / len(v), 1), "ic95": wilson(int(c), len(v))} for k, c in v.value_counts().items()}}
P5

{'archivo': 'revision_manual_pot_revisado.csv',
 'revisados': 50,
 'por_veredicto': {'phishing': {'n': 21,
   'pct': 42.0,
   'ic95': (np.float64(29.4), np.float64(55.8))},
  'spam': {'n': 17, 'pct': 34.0, 'ic95': (np.float64(22.4), np.float64(47.8))},
  'estafa': {'n': 11,
   'pct': 22.0,
   'ic95': (np.float64(12.8), np.float64(35.2))},
  'legitimo': {'n': 1,
   'pct': 2.0,
   'ic95': (np.float64(0.4), np.float64(10.5))}}}

In [8]:
# Celda 7: RESUMEN
RESUMEN = {"entorno": ENV, "P1_pruebas": P1, "carga_s": CARGA_S, "P2_fidelidad": P2, "P3_contrato": P3, "P4_latencia": P4,
           "P5_revision_manual": P5,
           "aceptacion": {"P2": bool(P2["pct_texto_modelo_identico"] >= 99 and P2["pct_misma_clase_que_07"] >= 98 and P2["dP_mediana"] < 0.01),
                          "P3": bool(P3["invalidas_contra_esquema"] == 0 and P3["con_ioc_vivo"] == 0), "P4": bool(P4["cumple_<1000ms"])}}
(RES / "inferencia_v1.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (08) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (08) =====
{"entorno":"colab","P1_pruebas":"363 passed, 1 skipped in 9.06s","carga_s":31.3,"P2_fidelidad":{"n":322,"pct_texto_modelo_identico":100.0,"pct_n_urls_identico":97.2,"pct_misma_clase_que_07":100.0,"dP_mediana":0.0,"dP_p95":0.0,"acierto_vs_etiqueta_pct":99.7},"P3_contrato":{"salidas":322,"invalidas_contra_esquema":0,"con_ioc_vivo":0},"P4_latencia":{"n":100,"media_ms":200.3,"p95_ms":348.2,"max_ms":387.0,"cumple_<1000ms":true},"P5_revision_manual":{"archivo":"revision_manual_pot_revisado.csv","revisados":50,"por_veredicto":{"phishing":{"n":21,"pct":42.0,"ic95":[29.4,55.8]},"spam":{"n":17,"pct":34.0,"ic95":[22.4,47.8]},"estafa":{"n":11,"pct":22.0,"ic95":[12.8,35.2]},"legitimo":{"n":1,"pct":2.0,"ic95":[0.4,10.5]}}},"aceptacion":{"P2":true,"P3":true,"P4":true}}
